# GPT-2 Medium/Large Two-Stage Fine-Tuning + Validation
This notebook trains `gpt2-medium` and `gpt2-large` in two stages:
1) QA pre-fine-tune on a selected QA dataset
2) task fine-tune on CommonsenseQA train split in your `Q:/Choices:/Answer:` prompt format

Then it evaluates each stage on validation sets and prints/saves metrics.


In [1]:
import gc
import json
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments, set_seed

from src.data.load_csqa import load_csqa

In [2]:
MODELS = ["gpt2-medium", "gpt2-large"]

STAGE1_DATASET = "openbookqa"
STAGE1_TRAIN_SPLIT = "train"
STAGE1_VAL_SPLIT = "validation"
STAGE1_TRAIN_LIMIT = None
STAGE1_VAL_LIMIT = None

CSQA_TRAIN_SPLIT = "train"
CSQA_VAL_SPLIT = "validation"
CSQA_TRAIN_LIMIT = None
CSQA_VAL_LIMIT = None

MAX_SEQ_LEN = "auto"  # int or "auto"

STAGE1_EPOCHS = 1.0
STAGE2_EPOCHS = 2.0
LEARNING_RATE = 5e-5
WARMUP_RATIO = 0.03
WEIGHT_DECAY = 0.01

DEFAULT_BATCH_SIZE = 2
DEFAULT_GRAD_ACCUM = 8
MODEL_BATCHING = {
    "gpt2-medium": {"batch_size": 2, "grad_accum": 8},
    "gpt2-large": {"batch_size": 1, "grad_accum": 16},
}

EVAL_BATCH_SIZE = 8
LOGGING_STEPS = 50
SEED = 123

USE_FP16 = True
USE_BF16 = True
GRADIENT_CHECKPOINTING = True

REPO_ROOT = Path.cwd().resolve()
OUTPUT_ROOT = REPO_ROOT / "checkpoints"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("device:", DEVICE)
print("repo root:", REPO_ROOT)
print("checkpoints root:", OUTPUT_ROOT)

device: cpu
repo root: C:\Users\mikol\OneDrive\Pulpit\Praca Magisterska\Transformer-Decision-Traces
checkpoints root: C:\Users\mikol\OneDrive\Pulpit\Praca Magisterska\Transformer-Decision-Traces\checkpoints


In [3]:
LETTERS = ["A", "B", "C", "D", "E"]


def now_id():
    return time.strftime("%Y%m%d-%H%M%S")


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    set_seed(seed)


def bf16_is_available():
    if not torch.cuda.is_available():
        return False
    if not hasattr(torch.cuda, "is_bf16_supported"):
        return False
    return bool(torch.cuda.is_bf16_supported())


def resolve_max_seq_len(tok, rows, max_seq_len, model_ctx):
    if isinstance(max_seq_len, str) and max_seq_len.strip().lower() == "auto":
        max_prompt_len = 0
        for row in rows:
            ids = tok(row["text"], add_special_tokens=False, truncation=False)["input_ids"]
            max_prompt_len = max(max_prompt_len, len(ids))
        # +1 because training appends the answer letter token
        needed = max_prompt_len + 1
        return int(min(model_ctx, needed))

    if max_seq_len is None:
        return int(model_ctx)

    val = int(max_seq_len)
    if val < 2:
        raise ValueError(f"MAX_SEQ_LEN must be >= 2, got {val}")
    return int(min(val, model_ctx))


def format_prompt(question, choices):
    lines = [f"Q: {question}", "Choices:"]
    for ch in choices:
        lines.append(f"{ch['label']}: {ch['text']}")
    lines.append("Answer:")
    return "\n".join(lines)


def load_csqa_rows(split, limit=None):
    df = load_csqa(split=split, limit=limit).reset_index(drop=True)
    rows = []
    for _, row in df.iterrows():
        rows.append(
            {
                "example_id": row["example_id"],
                "text": row["text"],
                "answerKey": str(row["answerKey"]).strip(),
                "n_choices": int(len(row["csqa_choices"])),
                "dataset": "commonsense_qa",
            }
        )
    return rows


def _raw_stage1_dataset(name, split):
    if name == "openbookqa":
        return load_dataset("openbookqa", "main", split=split)
    if name == "ai2_arc_challenge":
        return load_dataset("ai2_arc", "ARC-Challenge", split=split)
    if name == "ai2_arc_easy":
        return load_dataset("ai2_arc", "ARC-Easy", split=split)
    raise ValueError(f"Unsupported STAGE1_DATASET: {name}")


def load_stage1_rows(name, split, limit=None):
    ds = _raw_stage1_dataset(name, split)
    n = len(ds) if limit is None else min(int(limit), len(ds))
    if limit is not None:
        ds = ds.select(range(n))

    rows = []
    for i, ex in enumerate(ds):
        if name == "openbookqa":
            question = str(ex["question_stem"]).strip()
        else:
            question = str(ex["question"]).strip()

        labels = [str(x).strip() for x in ex["choices"]["label"]]
        texts = [str(x).strip() for x in ex["choices"]["text"]]
        answer_raw = str(ex["answerKey"]).strip()

        if len(texts) < 2 or len(texts) > 5:
            continue

        if answer_raw in labels:
            correct_idx = labels.index(answer_raw)
        elif answer_raw in LETTERS[: len(texts)]:
            correct_idx = LETTERS.index(answer_raw)
        elif answer_raw.isdigit() and (1 <= int(answer_raw) <= len(texts)):
            correct_idx = int(answer_raw) - 1
        else:
            continue

        normalized_choices = [{"label": LETTERS[j], "text": texts[j]} for j in range(len(texts))]
        rows.append(
            {
                "example_id": f"{name}:{split}:{i}",
                "text": format_prompt(question, normalized_choices),
                "answerKey": LETTERS[int(correct_idx)],
                "n_choices": int(len(texts)),
                "dataset": name,
            }
        )

    return rows

In [4]:
def build_answer_token_ids(tok):
    token_ids = {}
    for letter in LETTERS:
        ids = tok(" " + letter, add_special_tokens=False)["input_ids"]
        if len(ids) != 1:
            raise ValueError(f"Answer token '{letter}' is not single-token: {ids}")
        token_ids[letter] = int(ids[0])
    return token_ids


def _left_pad_to_len(seq, pad_id, max_len):
    seq = list(seq)
    if len(seq) > max_len:
        seq = seq[-max_len:]
    pad_len = max_len - len(seq)
    input_ids = [pad_id] * pad_len + seq
    attention_mask = [0] * pad_len + [1] * len(seq)
    return input_ids, attention_mask


class CausalAnswerTrainDataset(Dataset):
    def __init__(self, rows, tok, answer_token_ids, max_seq_len):
        self.input_ids = []
        self.attention_mask = []
        self.labels = []

        pad_id = tok.pad_token_id
        if pad_id is None:
            raise ValueError("Tokenizer must have pad_token_id")

        for row in rows:
            prompt_ids = tok(row["text"], add_special_tokens=False, truncation=False)["input_ids"]
            prompt_ids = prompt_ids[-(max_seq_len - 1):]
            y = int(answer_token_ids[row["answerKey"]])
            seq = prompt_ids + [y]
            ids, mask = _left_pad_to_len(seq, pad_id, max_seq_len)

            labels = [-100] * max_seq_len
            labels[-1] = y

            self.input_ids.append(ids)
            self.attention_mask.append(mask)
            self.labels.append(labels)

        self.input_ids = torch.tensor(self.input_ids, dtype=torch.long)
        self.attention_mask = torch.tensor(self.attention_mask, dtype=torch.long)
        self.labels = torch.tensor(self.labels, dtype=torch.long)

    def __len__(self):
        return int(self.input_ids.shape[0])

    def __getitem__(self, idx):
        return {
            "input_ids": self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
            "labels": self.labels[idx],
        }

class PromptEvalDataset(Dataset):
    def __init__(self, rows, tok, answer_token_ids, max_seq_len):
        self.input_ids = []
        self.attention_mask = []
        self.target_id = []
        self.choice_count = []

        pad_id = tok.pad_token_id
        if pad_id is None:
            raise ValueError("Tokenizer must have pad_token_id")

        for row in rows:
            prompt_ids = tok(row["text"], add_special_tokens=False, truncation=False)["input_ids"]
            prompt_ids = prompt_ids[-max_seq_len:]
            ids, mask = _left_pad_to_len(prompt_ids, pad_id, max_seq_len)

            self.input_ids.append(ids)
            self.attention_mask.append(mask)
            self.target_id.append(int(answer_token_ids[row["answerKey"]]))
            self.choice_count.append(int(row["n_choices"]))

        self.input_ids = torch.tensor(self.input_ids, dtype=torch.long)
        self.attention_mask = torch.tensor(self.attention_mask, dtype=torch.long)
        self.target_id = torch.tensor(self.target_id, dtype=torch.long)
        self.choice_count = torch.tensor(self.choice_count, dtype=torch.long)

    def __len__(self):
        return int(self.input_ids.shape[0])

    def __getitem__(self, idx):
        return {
            "input_ids": self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
            "target_id": self.target_id[idx],
            "choice_count": self.choice_count[idx],
        }


class StackCollator:
    def __call__(self, features):
        keys = features[0].keys()
        return {k: torch.stack([f[k] for f in features], dim=0) for k in keys}

In [5]:


def load_model_and_tokenizer(model_id_or_path):
    tok = AutoTokenizer.from_pretrained(model_id_or_path, use_fast=True)
    if tok.pad_token_id is None and tok.eos_token_id is not None:
        tok.pad_token = tok.eos_token
    tok.padding_side = "left"

    model = AutoModelForCausalLM.from_pretrained(model_id_or_path, attn_implementation="eager")
    if tok.pad_token_id is not None:
        model.config.pad_token_id = tok.pad_token_id
    model.to(DEVICE)
    model.train()
    return tok, model


def train_stage(
    model,
    tok,
    rows,
    answer_token_ids,
    max_seq_len,
    stage_name,
    out_dir,
    epochs,
    batch_size,
    grad_accum,
    learning_rate,
    warmup_ratio,
    weight_decay,
    logging_steps,
    seed,
):
    train_ds = CausalAnswerTrainDataset(rows, tok, answer_token_ids, max_seq_len)

    use_fp16 = bool(USE_FP16 and torch.cuda.is_available() and not bf16_is_available())
    use_bf16 = bool(USE_BF16 and bf16_is_available())

    if GRADIENT_CHECKPOINTING:
        model.gradient_checkpointing_enable()
        model.config.use_cache = False

    args = TrainingArguments(
        output_dir=str(out_dir),
        per_device_train_batch_size=int(batch_size),
        gradient_accumulation_steps=int(grad_accum),
        learning_rate=float(learning_rate),
        num_train_epochs=float(epochs),
        warmup_ratio=float(warmup_ratio),
        weight_decay=float(weight_decay),
        eval_strategy="no",
        save_strategy="epoch",
        save_total_limit=2,
        logging_steps=int(logging_steps),
        report_to="none",
        seed=int(seed),
        fp16=use_fp16,
        bf16=use_bf16,
        remove_unused_columns=False,
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        data_collator=StackCollator(),
        processing_class=tok,
    )

    train_metrics = trainer.train().metrics
    trainer.save_model(str(out_dir))
    tok.save_pretrained(str(out_dir))

    out = {k: float(v) for k, v in train_metrics.items() if isinstance(v, (int, float))}
    out["stage"] = stage_name
    out["train_examples"] = int(len(rows))
    out["output_dir"] = str(out_dir)

    del trainer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

    return out

@torch.no_grad()
def evaluate_mc(model, tok, rows, answer_token_ids, batch_size, max_seq_len):
    model.eval()

    ds = PromptEvalDataset(rows, tok, answer_token_ids, max_seq_len)
    dl = DataLoader(ds, batch_size=int(batch_size), shuffle=False, collate_fn=StackCollator())

    option_token_ids = [int(answer_token_ids[l]) for l in LETTERS]

    total = 0
    correct = 0
    true_probs = []
    pred_probs = []
    entropies = []
    margins = []

    for batch in dl:
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        target_ids = batch["target_id"]
        choice_counts = batch["choice_count"]

        out = model(input_ids=input_ids, attention_mask=attention_mask, return_dict=True)
        logits = out.logits

        bsz = int(input_ids.shape[0])
        for i in range(bsz):
            nz = torch.nonzero(attention_mask[i], as_tuple=False).view(-1)
            if int(nz.numel()) == 0:
                continue

            pos = int(nz[-1].item())
            vec = logits[i, pos].float().detach().cpu()

            c = int(choice_counts[i].item())
            valid_ids = option_token_ids[:c]

            true_id = int(target_ids[i].item())
            if true_id not in valid_ids:
                continue

            choice_logits = vec[valid_ids]
            probs = torch.softmax(choice_logits, dim=-1)

            pred_local = int(torch.argmax(choice_logits).item())
            pred_id = int(valid_ids[pred_local])
            true_local = int(valid_ids.index(true_id))

            total += 1
            if pred_id == true_id:
                correct += 1

            true_probs.append(float(probs[true_local].item()))
            pred_probs.append(float(probs[pred_local].item()))
            entropies.append(float((-(probs * torch.log(probs + 1e-12))).sum().item()))

            if c >= 2:
                sorted_vals, _ = torch.sort(choice_logits, descending=True)
                margins.append(float((sorted_vals[0] - sorted_vals[1]).item()))

    acc = float(correct / total) if total > 0 else float("nan")
    return {
        "n": int(total),
        "acc": acc,
        "true_prob_mean": float(np.mean(true_probs)) if true_probs else float("nan"),
        "pred_prob_mean": float(np.mean(pred_probs)) if pred_probs else float("nan"),
        "choice_entropy_mean": float(np.mean(entropies)) if entropies else float("nan"),
        "margin_mean": float(np.mean(margins)) if margins else float("nan"),
    }

In [6]:
seed_everything(SEED)

csqa_train_rows = load_csqa_rows(CSQA_TRAIN_SPLIT, CSQA_TRAIN_LIMIT)
csqa_val_rows = load_csqa_rows(CSQA_VAL_SPLIT, CSQA_VAL_LIMIT)

if STAGE1_DATASET is None:
    stage1_train_rows = []
    stage1_val_rows = []
else:
    stage1_train_rows = load_stage1_rows(STAGE1_DATASET, STAGE1_TRAIN_SPLIT, STAGE1_TRAIN_LIMIT)
    stage1_val_rows = load_stage1_rows(STAGE1_DATASET, STAGE1_VAL_SPLIT, STAGE1_VAL_LIMIT)

print("csqa_train:", len(csqa_train_rows))
print("csqa_val:", len(csqa_val_rows))
print("stage1_train:", len(stage1_train_rows))
print("stage1_val:", len(stage1_val_rows))

if len(csqa_train_rows) == 0 or len(csqa_val_rows) == 0:
    raise RuntimeError("CSQA rows are empty")

README.md: 0.00B [00:00, ?B/s]

c:\Users\mikol\venvs\Masters_env\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\mikol\.cache\huggingface\hub\datasets--openbookqa. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


main/train-00000-of-00001.parquet:   0%|          | 0.00/496k [00:00<?, ?B/s]

main/validation-00000-of-00001.parquet:   0%|          | 0.00/58.2k [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/55.5k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/4957 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/500 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/500 [00:00<?, ? examples/s]

csqa_train: 9741
csqa_val: 1221
stage1_train: 4957
stage1_val: 500


In [ ]:
run_id = now_id()
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

all_metrics = []
train_logs = []
model_artifacts = []

for model_name in MODELS:
    print("=" * 80)
    print("model:", model_name)

    batch_cfg = MODEL_BATCHING.get(model_name, {})
    train_batch = int(batch_cfg.get("batch_size", DEFAULT_BATCH_SIZE))
    grad_accum = int(batch_cfg.get("grad_accum", DEFAULT_GRAD_ACCUM))

    tok, model = load_model_and_tokenizer(model_name)
    answer_token_ids = build_answer_token_ids(tok)

    model_ctx = int(getattr(model.config, "n_positions", getattr(model.config, "n_ctx", 1024)))
    max_seq_len = resolve_max_seq_len(
        tok=tok,
        rows=(csqa_train_rows + csqa_val_rows + stage1_train_rows + stage1_val_rows),
        max_seq_len=MAX_SEQ_LEN,
        model_ctx=model_ctx,
    )
    print("max_seq_len:", max_seq_len, "(model_ctx:", model_ctx, ")")

    base_csqa = evaluate_mc(model, tok, csqa_val_rows, answer_token_ids, EVAL_BATCH_SIZE, max_seq_len)
    all_metrics.append(
        {
            "model": model_name,
            "stage": "base",
            "dataset": "csqa_validation",
            **base_csqa,
        }
    )

    if len(stage1_val_rows) > 0:
        base_stage1 = evaluate_mc(model, tok, stage1_val_rows, answer_token_ids, EVAL_BATCH_SIZE, max_seq_len)
        all_metrics.append(
            {
                "model": model_name,
                "stage": "base",
                "dataset": f"{STAGE1_DATASET}_validation",
                **base_stage1,
            }
        )

    if STAGE1_DATASET is not None and len(stage1_train_rows) > 0:
        stage1_dir = OUTPUT_ROOT / f"{run_id}_{model_name}_stage1_{STAGE1_DATASET}".replace("/", "-")
        log1 = train_stage(
            model=model,
            tok=tok,
            rows=stage1_train_rows,
            answer_token_ids=answer_token_ids,
            max_seq_len=max_seq_len,
            stage_name="stage1",
            out_dir=stage1_dir,
            epochs=STAGE1_EPOCHS,
            batch_size=train_batch,
            grad_accum=grad_accum,
            learning_rate=LEARNING_RATE,
            warmup_ratio=WARMUP_RATIO,
            weight_decay=WEIGHT_DECAY,
            logging_steps=LOGGING_STEPS,
            seed=SEED,
        )
        train_logs.append({"model": model_name, **log1})

        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

        tok, model = load_model_and_tokenizer(str(stage1_dir))
        answer_token_ids = build_answer_token_ids(tok)

        stage1_csqa = evaluate_mc(model, tok, csqa_val_rows, answer_token_ids, EVAL_BATCH_SIZE, max_seq_len)
        all_metrics.append(
            {
                "model": model_name,
                "stage": "after_stage1",
                "dataset": "csqa_validation",
                **stage1_csqa,
            }
        )

        if len(stage1_val_rows) > 0:
            stage1_val = evaluate_mc(model, tok, stage1_val_rows, answer_token_ids, EVAL_BATCH_SIZE, max_seq_len)
            all_metrics.append(
                {
                    "model": model_name,
                    "stage": "after_stage1",
                    "dataset": f"{STAGE1_DATASET}_validation",
                    **stage1_val,
                }
            )

    stage2_dir = OUTPUT_ROOT / f"{run_id}_{model_name}_stage2_csqa".replace("/", "-")
    log2 = train_stage(
        model=model,
        tok=tok,
        rows=csqa_train_rows,
        answer_token_ids=answer_token_ids,
        max_seq_len=max_seq_len,
        stage_name="stage2",
        out_dir=stage2_dir,
        epochs=STAGE2_EPOCHS,
        batch_size=train_batch,
        grad_accum=grad_accum,
        learning_rate=LEARNING_RATE,
        warmup_ratio=WARMUP_RATIO,
        weight_decay=WEIGHT_DECAY,
        logging_steps=LOGGING_STEPS,
        seed=SEED,
    )
    train_logs.append({"model": model_name, **log2})

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

    tok, model = load_model_and_tokenizer(str(stage2_dir))
    answer_token_ids = build_answer_token_ids(tok)

    final_csqa = evaluate_mc(model, tok, csqa_val_rows, answer_token_ids, EVAL_BATCH_SIZE, max_seq_len)
    all_metrics.append(
        {
            "model": model_name,
            "stage": "after_stage2_csqa",
            "dataset": "csqa_validation",
            **final_csqa,
        }
    )

    if len(stage1_val_rows) > 0:
        final_stage1 = evaluate_mc(model, tok, stage1_val_rows, answer_token_ids, EVAL_BATCH_SIZE, max_seq_len)
        all_metrics.append(
            {
                "model": model_name,
                "stage": "after_stage2_csqa",
                "dataset": f"{STAGE1_DATASET}_validation",
                **final_stage1,
            }
        )

    model_artifacts.append(
        {
            "model": model_name,
            "stage2_checkpoint": str(stage2_dir),
        }
    )

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

metrics_df = pd.DataFrame(all_metrics)
train_df = pd.DataFrame(train_logs)

metrics_df

In [ ]:
assert len(metrics_df) > 0, "No metrics collected."

display(metrics_df.sort_values(["model", "dataset", "stage"]).reset_index(drop=True))

pivot_acc = metrics_df.pivot_table(index=["model", "stage"], columns="dataset", values="acc")
display(pivot_acc)

display(train_df.sort_values(["model", "stage"]).reset_index(drop=True))

display(pd.DataFrame(model_artifacts))
